In [62]:
# ======================================================================
# DUBLIN BER DATA (ready to use)
# ======================================================================
# Input:  the 19 national processed BER tables
#         (dataSet/processed_dataset/housing_energy_dataset)
#
# What this notebook does so far:
#   1. loads every CSV file in the BER folder
#   2. keeps only the Dublin homes
#   3. removes the columns that are the same thing as the target (BerRating)
# ======================================================================

import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

In [63]:
# Folder paths

base = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project'

ber_folder = os.path.join(base, 'dataSet', 'processed_dataset', 'housing_energy_dataset')

print(ber_folder)

C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project\dataSet\processed_dataset\housing_energy_dataset


In [64]:
# Which CSV files are in the BER folder?

csv_files = []
for file in sorted(os.listdir(ber_folder)):
    if file.endswith('.csv'):
        csv_files.append(file)

print('Number of CSV files:', len(csv_files))
csv_files

Number of CSV files: 19


['Final_datset_assessment.csv',
 'Final_datset_county.csv',
 'Final_datset_dwelling_type.csv',
 'Final_datset_fabric.csv',
 'Final_datset_floors.csv',
 'Final_datset_fuel.csv',
 'Final_datset_geometry.csv',
 'Final_datset_group_boilers.csv',
 'Final_datset_group_heating.csv',
 'Final_datset_nzeb_renewables.csv',
 'Final_datset_other_energy.csv',
 'Final_datset_results.csv',
 'Final_datset_solar_pv.csv',
 'Final_datset_solar_water.csv',
 'Final_datset_space_heating.csv',
 'Final_datset_ventilation.csv',
 'Final_datset_walls.csv',
 'Final_datset_water_heating.csv',
 'Final_datset_wind_turbines.csv']

In [65]:
# Loading every CSV file into one dictionary
# 'Final_datset_results.csv' -> tables['results']

tables = {}

for file in csv_files:
    name = file.replace('Final_datset_', '').replace('.csv', '')
    tables[name] = pd.read_csv(os.path.join(ber_folder, file), low_memory=False)
    print(name, tables[name].shape)

assessment (569058, 12)
county (52, 2)
dwelling_type (10, 2)
fabric (569058, 15)
floors (975905, 4)
fuel (18, 2)
geometry (569058, 12)
group_boilers (37003, 5)
group_heating (569058, 17)
nzeb_renewables (569058, 8)
other_energy (124817, 13)
results (569058, 25)
solar_pv (182, 7)
solar_water (569058, 15)
space_heating (569058, 22)
ventilation (569058, 9)
walls (952212, 9)
water_heating (569058, 22)
wind_turbines (1, 5)


In [66]:
# space_heating and water_heating both have a column called 'main_fuel_id'.
# Renaming them so they don't clash when we join.

tables['space_heating'] = tables['space_heating'].rename(columns={'main_fuel_id': 'sh_main_fuel_id'})
tables['water_heating'] = tables['water_heating'].rename(columns={'main_fuel_id': 'wh_main_fuel_id'})

In [67]:
# Joining the tables that have ONE row per home, using ber_id.
# We start from the assessment table and add the others one by one.

one_row_tables = ['results', 'geometry', 'fabric', 'ventilation', 'space_heating',
                  'water_heating', 'solar_water', 'group_heating', 'nzeb_renewables']

df_ber = tables['assessment']

for name in one_row_tables:
    df_ber = df_ber.merge(tables[name], on='ber_id', how='left')
    print('after', name, df_ber.shape)

after results (569058, 36)
after geometry (569058, 47)
after fabric (569058, 61)
after ventilation (569058, 69)
after space_heating (569058, 90)
after water_heating (569058, 111)
after solar_water (569058, 125)
after group_heating (569058, 141)
after nzeb_renewables (569058, 148)


In [68]:
# These 6 tables have SEVERAL rows per home (e.g. one row per wall).
# If we joined them directly, a home would appear many times.
# So first we make one summary row per home.

floors_summary = tables['floors'].groupby('ber_id').agg(
    num_floors=('floor_area', 'count'),
    total_floor_area=('floor_area', 'sum'))

walls_summary = tables['walls'].groupby('ber_id').agg(
    num_walls=('wall_area', 'count'),
    total_wall_area=('wall_area', 'sum'),
    mean_wall_uvalue=('wall_uvalue', 'mean'))

other_energy_summary = tables['other_energy'].groupby('ber_id').agg(
    num_other_energy=('energy_type', 'count'))

group_boilers_summary = tables['group_boilers'].groupby('ber_id').agg(
    num_group_boilers=('plant_efficiency', 'count'))

solar_pv_summary = tables['solar_pv'].groupby('ber_id').agg(
    num_pv_arrays=('no_panels', 'count'),
    total_pv_panels=('no_panels', 'sum'))

wind_summary = tables['wind_turbines'].groupby('ber_id').agg(
    num_wind_turbines=('rotor_diameter', 'count'))

In [69]:
# Joining the summaries.
# A home with no row in a table (e.g. no solar panels) gets count 0, not empty.

summaries = [floors_summary, walls_summary, other_energy_summary,
             group_boilers_summary, solar_pv_summary, wind_summary]

for summary in summaries:
    df_ber = df_ber.merge(summary, on='ber_id', how='left')

count_columns = ['num_floors', 'num_walls', 'num_other_energy',
                 'num_group_boilers', 'num_pv_arrays', 'num_wind_turbines']

df_ber[count_columns] = df_ber[count_columns].fillna(0)

In [70]:
# The 3 lookup tables turn id numbers into names
# (county_id 6 -> 'Co. Dublin', fuel_id 9 -> 'Mains Gas', ...)

county_names = dict(zip(tables['county']['county_id'], tables['county']['CountyName']))
dwelling_names = dict(zip(tables['dwelling_type']['dwelling_type_id'], tables['dwelling_type']['DwellingTypeDescr']))
fuel_names = dict(zip(tables['fuel']['fuel_id'], tables['fuel']['FuelName']))

df_ber['CountyName'] = df_ber['county_id'].map(county_names)
df_ber['DwellingTypeDescr'] = df_ber['dwelling_type_id'].map(dwelling_names)
df_ber['sh_main_fuel'] = df_ber['sh_main_fuel_id'].map(fuel_names)
df_ber['wh_main_fuel'] = df_ber['wh_main_fuel_id'].map(fuel_names)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_24692\1977224985.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ber['CountyName'] = df_ber['county_id'].map(county_names)
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_24692\1977224985.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ber['DwellingTypeDescr'] = df_ber['dwelling_type_id'].map(dwelling_names)
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_24692\1977224985.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.i

In [71]:
# Check: one row per home, nothing lost

print('Rows:', df_ber.shape[0], ' Columns:', df_ber.shape[1])
print('Homes in assessment:', len(tables['assessment']))
print('Duplicate ber_id:', df_ber['ber_id'].duplicated().sum())   # should be 0

df_ber.head()

Rows: 569058  Columns: 162
Homes in assessment: 569058
Duplicate ber_id: 0


,ber_id,Year_of_Construction,TypeofRating,PurposeOfRating,DateOfAssessment,MultiDwellingMPRN,TGDLEdition,SA_Code,prob_smarea_error_0corr,prob_smarea_error_100corr,county_id,dwelling_type_id,BerRating,EnergyRating,CO2Rating,MPCDERValue,RER,RenewEPnren,RenewEPren,CPC,EPC,GlobalWarmingPotentialValue,HeatLossIndicator,SHMainPrimaryEnergy,SHSecPrimaryEnergy,WHMainPrimaryEnergy,WHSecPrimaryEnergy,PumpsFansPrimaryEnergy,LightingPrimaryEnergy,CoolingPrimaryEnergy,CHPInputPrimaryEnergy,CHPElecOutputPrimaryEnergy,RenewPrimaryEnergy1,RenewPrimaryEnergy2,RenewPrimaryEnergy3,RenewPrimaryEPTot,GroundFloorArea(sq m),NoStoreys,WallArea,RoofArea,FloorArea,WindowArea,DoorArea,LivingAreaPercent,RoomInRoofArea,PredominantRoofType,PredominantRoofTypeArea,UValueWall,UValueRoof,UValueFloor,UValueWindow,UvalueDoor,GroundFloorUValue,ThermalBridgingFactor,ThermalMassCategory,StructureType,SuspendedWoodenFloor,PercentageDraughtStripped,NoOfSidesSheltered,PermeabilityTest,PermeabilityTestResult,NoOfChimneys,NoOfOpenFlues,NoOfFansAndVents,NoOfFluelessGasFires,DraftLobby,VentilationMethod,FanPowerManuDeclaredValue,HeatExchangerEff,HSMainSystemEfficiency,HSEffAdjFactor,HSSupplHeatFraction,HSSupplSystemEff,SupplSHFuel,SupplHSFuelTypeID,SHRenewableResources,TempAdjustment,HeatSystemControlCat,HeatSystemResponseCat,NoCentralHeatingPumps,CHBoilerThermostatControlled,NoOilBoilerHeatingPumps,OBBoilerThermostatControlled,OBPumpInsideDwelling,NoGasBoilerHeatingPumps,WarmAirHeatingSystem,UndergroundHeating,CombiBoiler,CapacityToWorkAtLowTemperatureLevels,sh_main_fuel_id,WHMainSystemEff,WHEffAdjFactor,SupplWHFuel,WHRenewableResources,StorageLosses,ManuLossFactorAvail,ElecImmersionInSummer,KeepHotFacility,WaterStorageVolume,DeclaredLossFactor,TempFactorUnadj,TempFactorMultiplier,InsulationType,InsulationThickness,PrimaryCircuitLoss,CombiBoilerAddLoss,ElecConsumpKeepHot,CylinderStat,CombinedCylinder,VolumeOfPreHeatStore,wh_main_fuel_id,SolarHotWaterHeating,ApertureArea,ZeroLossCollectorEff,CollectorHeatLossCoEff,AnnualSolarRadiation,OvershadingFactor,SolarStorageVolume,ElectricityConsumption,SWHPumpSolarPowered,SolarHeatFraction,SolarSpaceHeatingSystem,SolarWaterHeatingOrientation,SolarWaterHeatingTiltInDegreesFromHorizontal,SolarWaterHeatingOvershading,ChargingBasisHeatConsumed,gsdHSSupplHeatFraction,gsdHSSupplSystemEff,DistLossFactor,CHPUnitHeatFraction,CHPSystemType,CHPSystemTypeDescription,GroupHeatingCO2EQEmissionsFactor,GroupHeatingNonRenewablePEF,GroupHeatingRenewablePEF,HeatLossFromDistributionSystemOutsideDwelling,CHPElecEff,CHPHeatEff,CHPFuelType,gsdSHRenewableResources,gsdWHRenewableResources,IsSolarPVBatteryPresent,SolarBatteryCapacity,SolarPVEnergyOutput,WindTurbineEnergyOutput,IsPropertyExemptFromZeb,OverrideIsFossilFuelUsedOnSite,IsSystemApplicable,num_floors,total_floor_area,num_walls,total_wall_area,mean_wall_uvalue,num_other_energy,num_group_boilers,num_pv_arrays,total_pv_panels,num_wind_turbines,CountyName,DwellingTypeDescr,sh_main_fuel,wh_main_fuel
0,1,2000,Existing,Private Letting,2021-01-01,YES,0,NaN,NaN,NaN,9,3,339.72,E1,79.15,65.66,0.000,8275.79,0.0,1.217,1.056,NaN,NaN,4324.60,0.00,1932.40,929.89,1348.62,178.41,NaN,0.0,0.0,0.0,0.0,0.0,8275.79,25.65,1,50.75,25.65,25.65,3.47,1.85,45.61,0.0,Pitch Roof-Insul.on Rafter,25.65,0.55,0.36,0.52,2.80,3.00,0.0,0.15,Medium,Masonry,No,100.0,2.0,NO,0.0,0.0,0.0,1.0,0.0,YES,Natural vent.,0.00,0.0,85.60,0.95,0.0,0.0,1.0,NaN,1.0,0.0,1,1,1,YES,1,NO,NO,0,NO,NO,None,NaN,7.0,85.60,0.95,28.0,1.0,YES,NO,YES,inapplicable,110.0,0.0,0.6,1.3,Factory Insulated,30.0,Boiler with uninsulated primar,0.0,0.0,NO,NO,NaN,7.0,NO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO,NaN,NaN,NaN,NaN,NaN,NO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.00,0.0,NaN,NaN,NaN,1.0,25.65,1.0,50.75,0.5500,0.0,0.0,0.0,NaN,0.0,Co. Kildare,Detached house,Heating Oil,Heating Oil
1,2,2000,Existing,Private Letting,2021-01-01,YES,0,NaN,NaN,NaN,9,3,290.10,D2,68.41,51.77,0.000,11375.40,0.0,1.314,1.136,NaN,NaN,7108.39,0.00,2057.21,989.85,1453.54,232.77,

In [72]:
# Keeping only the Dublin homes and saving them as df_dublin_ber.
# Dublin is written as 'Co. Dublin' or as a postal district ('Dublin 1' ... 'Dublin 24'),
# so we keep every row where the county name contains 'Dublin'.

df_dublin_ber = df_ber[df_ber['CountyName'].str.contains('Dublin')].copy()

print('All of Ireland:', len(df_ber))
print('Dublin only:   ', len(df_dublin_ber))
df_dublin_ber['CountyName'].value_counts()

# df_dublin_ber.head()

All of Ireland: 569058
Dublin only:    181195


CountyName
Co. Dublin    48327
Dublin 15     13465
Dublin 18     12075
Dublin 24     11841
Dublin 12      7457
Dublin 8       7260
Dublin 11      7119
Dublin 22      6906
Dublin 9       6903
Dublin 7       6265
Dublin 13      5829
Dublin 4       5789
Dublin 16      5670
Dublin 14      5362
Dublin 3       5120
Dublin 5       5034
Dublin 1       4556
Dublin 6       4106
Dublin 6W      2808
Dublin 10      2631
Dublin 17      2363
Dublin 2       2246
Dublin 20      2063
Name: count, dtype: int64

In [73]:
# Columns that are the same thing as our target (BerRating).
# The BER software calculates them from the same energy result, so they are not
# building features, and a model would "cheat" with them. We remove them.

same_as_target = ['CO2Rating', 'EPC', 'CPC', 'HeatLossIndicator',
                  'RER', 'RenewEPnren', 'RenewEPren', 'MPCDERValue', 'GlobalWarmingPotentialValue',
                  'SHMainPrimaryEnergy', 'SHSecPrimaryEnergy',
                  'WHMainPrimaryEnergy', 'WHSecPrimaryEnergy',
                  'PumpsFansPrimaryEnergy', 'LightingPrimaryEnergy', 'CoolingPrimaryEnergy',
                  'CHPInputPrimaryEnergy', 'CHPElecOutputPrimaryEnergy',
                  'RenewPrimaryEnergy1', 'RenewPrimaryEnergy2', 'RenewPrimaryEnergy3',
                  'RenewPrimaryEPTot']

print('Columns before:', df_dublin_ber.shape[1])
df_dublin_ber = df_dublin_ber.drop(columns=same_as_target)
print('Columns after: ', df_dublin_ber.shape[1])


Columns before: 162
Columns after:  140


In [74]:
df_dublin_ber.head()

,ber_id,Year_of_Construction,TypeofRating,PurposeOfRating,DateOfAssessment,MultiDwellingMPRN,TGDLEdition,SA_Code,prob_smarea_error_0corr,prob_smarea_error_100corr,county_id,dwelling_type_id,BerRating,EnergyRating,GroundFloorArea(sq m),NoStoreys,WallArea,RoofArea,FloorArea,WindowArea,DoorArea,LivingAreaPercent,RoomInRoofArea,PredominantRoofType,PredominantRoofTypeArea,UValueWall,UValueRoof,UValueFloor,UValueWindow,UvalueDoor,GroundFloorUValue,ThermalBridgingFactor,ThermalMassCategory,StructureType,SuspendedWoodenFloor,PercentageDraughtStripped,NoOfSidesSheltered,PermeabilityTest,PermeabilityTestResult,NoOfChimneys,NoOfOpenFlues,NoOfFansAndVents,NoOfFluelessGasFires,DraftLobby,VentilationMethod,FanPowerManuDeclaredValue,HeatExchangerEff,HSMainSystemEfficiency,HSEffAdjFactor,HSSupplHeatFraction,HSSupplSystemEff,SupplSHFuel,SupplHSFuelTypeID,SHRenewableResources,TempAdjustment,HeatSystemControlCat,HeatSystemResponseCat,NoCentralHeatingPumps,CHBoilerThermostatControlled,NoOilBoilerHeatingPumps,OBBoilerThermostatControlled,OBPumpInsideDwelling,NoGasBoilerHeatingPumps,WarmAirHeatingSystem,UndergroundHeating,CombiBoiler,CapacityToWorkAtLowTemperatureLevels,sh_main_fuel_id,WHMainSystemEff,WHEffAdjFactor,SupplWHFuel,WHRenewableResources,StorageLosses,ManuLossFactorAvail,ElecImmersionInSummer,KeepHotFacility,WaterStorageVolume,DeclaredLossFactor,TempFactorUnadj,TempFactorMultiplier,InsulationType,InsulationThickness,PrimaryCircuitLoss,CombiBoilerAddLoss,ElecConsumpKeepHot,CylinderStat,CombinedCylinder,VolumeOfPreHeatStore,wh_main_fuel_id,SolarHotWaterHeating,ApertureArea,ZeroLossCollectorEff,CollectorHeatLossCoEff,AnnualSolarRadiation,OvershadingFactor,SolarStorageVolume,ElectricityConsumption,SWHPumpSolarPowered,SolarHeatFraction,SolarSpaceHeatingSystem,SolarWaterHeatingOrientation,SolarWaterHeatingTiltInDegreesFromHorizontal,SolarWaterHeatingOvershading,ChargingBasisHeatConsumed,gsdHSSupplHeatFraction,gsdHSSupplSystemEff,DistLossFactor,CHPUnitHeatFraction,CHPSystemType,CHPSystemTypeDescription,GroupHeatingCO2EQEmissionsFactor,GroupHeatingNonRenewablePEF,GroupHeatingRenewablePEF,HeatLossFromDistributionSystemOutsideDwelling,CHPElecEff,CHPHeatEff,CHPFuelType,gsdSHRenewableResources,gsdWHRenewableResources,IsSolarPVBatteryPresent,SolarBatteryCapacity,SolarPVEnergyOutput,WindTurbineEnergyOutput,IsPropertyExemptFromZeb,OverrideIsFossilFuelUsedOnSite,IsSystemApplicable,num_floors,total_floor_area,num_walls,total_wall_area,mean_wall_uvalue,num_other_energy,num_group_boilers,num_pv_arrays,total_pv_panels,num_wind_turbines,CountyName,DwellingTypeDescr,sh_main_fuel,wh_main_fuel
13,14,1930,Existing,Grant Support,2021-01-03,NO,0,NaN,NaN,NaN,31,9,216.42,C3,67.050,1,54.27,67.05,67.05,9.21,2.18,35.44,0.0,Pitch Roof-Insul.on Ceiling,67.050,0.29,0.19,0.73,1.10,1.71,0.0,0.15,High,Masonry,No,88.0,2.0,NO,0.0,0.0,1.0,2.0,0.0,NO,Natural vent.,0.0,0.0,91.1,0.95,0.1,60.0,8.0,NaN,1.0,0.6,1,1,1,NO,0,NO,NO,1,NO,NO,"Instantaneous, without keep-ho",NaN,9.0,91.1,0.95,1.0,1.0,NO,NO,NO,inapplicable,0.0,0.0,0.0,0.0,None,0.0,Combi boiler,600.0,0.0,NO,NO,NaN,9.0,NO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO,NaN,NaN,NaN,NaN,NaN,NO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,1.0,67.050,2.0,54.270,0.280,0.0,0.0,0.0,NaN,0.0,Dublin 12,Semi-detached house,Mains Gas,Mains Gas
18,19,2005,Existing,Sale,2021-01-04,NO,0,NaN,NaN,NaN,30,8,132.19,B3,88.000,2,40.33,44.95,44.95,12.78,1.85,17.67,0.0,Pitch Roof-Insul.on Ceiling,43.050,0.37,0.26,0.26,2.00,3.00,0.0,0.15,Medium,Masonry,No,90.0,4.0,NO,0.0,1.0,0.0,1.0,0.0,NO,Natural vent.,0.0,0.0,90.1,0.95,0.1,30.0,8.0,NaN,1.0,0.0,2,1,1,YES,0,NO,NO,1,NO,NO,None,NaN,9.0,90.1,0.95,28.0,1.0,YES,NO,YES,inapplicable,110.0,0.0,0.6,1.0,Factory Insulated,30.0,Boiler with uninsulated primar,0.0,0.0,NO,NO,NaN,9.0,NO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO,NaN,NaN,NaN,NaN,NaN,NO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,2.0,88.000,1.0,40.330,0.370,0.0,0.0,0.0,NaN,0.0,Dublin 11,Mid-terr

In [75]:
df_dublin_ber.isnull().sum()

ber_id                      0
Year_of_Construction        0
TypeofRating                0
PurposeOfRating             0
DateOfAssessment            0
                        ...  
num_wind_turbines           0
CountyName                  0
DwellingTypeDescr           0
sh_main_fuel            11334
wh_main_fuel            11335
Length: 140, dtype: int64

In [76]:
# Missing values in each column, as a percentage of all Dublin homes

missing_percent = df_dublin_ber.isnull().sum() / len(df_dublin_ber) * 100

missing_percent.round(2).sort_values(ascending=False)

prob_smarea_error_0corr                 100.0
prob_smarea_error_100corr               100.0
SA_Code                                 100.0
CapacityToWorkAtLowTemperatureLevels    100.0
gsdSHRenewableResources                 100.0
                                        ...  
num_group_boilers                         0.0
num_wind_turbines                         0.0
num_pv_arrays                             0.0
DwellingTypeDescr                         0.0
CountyName                                0.0
Length: 140, dtype: float64

In [77]:
# Removing the columns where more than 50% of the values are missing

columns_to_remove = missing_percent[missing_percent > 50].index

print('Columns removed:', len(columns_to_remove))
print(list(columns_to_remove))

print('Columns before:', df_dublin_ber.shape[1])
df_dublin_ber = df_dublin_ber.drop(columns=columns_to_remove)
print('Columns after: ', df_dublin_ber.shape[1])

Columns removed: 37
['SA_Code', 'prob_smarea_error_0corr', 'prob_smarea_error_100corr', 'SupplHSFuelTypeID', 'CapacityToWorkAtLowTemperatureLevels', 'VolumeOfPreHeatStore', 'ApertureArea', 'ZeroLossCollectorEff', 'CollectorHeatLossCoEff', 'AnnualSolarRadiation', 'OvershadingFactor', 'SolarStorageVolume', 'ElectricityConsumption', 'SolarHeatFraction', 'SolarSpaceHeatingSystem', 'SolarWaterHeatingOrientation', 'SolarWaterHeatingTiltInDegreesFromHorizontal', 'SolarWaterHeatingOvershading', 'gsdHSSupplHeatFraction', 'gsdHSSupplSystemEff', 'DistLossFactor', 'CHPUnitHeatFraction', 'CHPSystemType', 'CHPSystemTypeDescription', 'GroupHeatingCO2EQEmissionsFactor', 'GroupHeatingNonRenewablePEF', 'GroupHeatingRenewablePEF', 'HeatLossFromDistributionSystemOutsideDwelling', 'CHPElecEff', 'CHPHeatEff', 'CHPFuelType', 'gsdSHRenewableResources', 'gsdWHRenewableResources', 'IsPropertyExemptFromZeb', 'OverrideIsFossilFuelUsedOnSite', 'IsSystemApplicable', 'total_pv_panels']
Columns before: 140
Columns af

In [78]:
df_dublin_ber.head(100)

,ber_id,Year_of_Construction,TypeofRating,PurposeOfRating,DateOfAssessment,MultiDwellingMPRN,TGDLEdition,county_id,dwelling_type_id,BerRating,EnergyRating,GroundFloorArea(sq m),NoStoreys,WallArea,RoofArea,FloorArea,WindowArea,DoorArea,LivingAreaPercent,RoomInRoofArea,PredominantRoofType,PredominantRoofTypeArea,UValueWall,UValueRoof,UValueFloor,UValueWindow,UvalueDoor,GroundFloorUValue,ThermalBridgingFactor,ThermalMassCategory,StructureType,SuspendedWoodenFloor,PercentageDraughtStripped,NoOfSidesSheltered,PermeabilityTest,PermeabilityTestResult,NoOfChimneys,NoOfOpenFlues,NoOfFansAndVents,NoOfFluelessGasFires,DraftLobby,VentilationMethod,FanPowerManuDeclaredValue,HeatExchangerEff,HSMainSystemEfficiency,HSEffAdjFactor,HSSupplHeatFraction,HSSupplSystemEff,SupplSHFuel,SHRenewableResources,TempAdjustment,HeatSystemControlCat,HeatSystemResponseCat,NoCentralHeatingPumps,CHBoilerThermostatControlled,NoOilBoilerHeatingPumps,OBBoilerThermostatControlled,OBPumpInsideDwelling,NoGasBoilerHeatingPumps,WarmAirHeatingSystem,UndergroundHeating,CombiBoiler,sh_main_fuel_id,WHMainSystemEff,WHEffAdjFactor,SupplWHFuel,WHRenewableResources,StorageLosses,ManuLossFactorAvail,ElecImmersionInSummer,KeepHotFacility,WaterStorageVolume,DeclaredLossFactor,TempFactorUnadj,TempFactorMultiplier,InsulationType,InsulationThickness,PrimaryCircuitLoss,CombiBoilerAddLoss,ElecConsumpKeepHot,CylinderStat,CombinedCylinder,wh_main_fuel_id,SolarHotWaterHeating,SWHPumpSolarPowered,ChargingBasisHeatConsumed,IsSolarPVBatteryPresent,SolarBatteryCapacity,SolarPVEnergyOutput,WindTurbineEnergyOutput,num_floors,total_floor_area,num_walls,total_wall_area,mean_wall_uvalue,num_other_energy,num_group_boilers,num_pv_arrays,num_wind_turbines,CountyName,DwellingTypeDescr,sh_main_fuel,wh_main_fuel
13,14,1930,Existing,Grant Support,2021-01-03,NO,0,31,9,216.42,C3,67.050,1,54.27,67.05,67.05,9.21,2.18,35.44,0.00,Pitch Roof-Insul.on Ceiling,67.050,0.29,0.19,0.73,1.10,1.71,0.0,0.15,High,Masonry,No,88.0,2.0,NO,0.00,0.0,1.0,2.0,0.0,NO,Natural vent.,0.00,0.0,91.1,0.95,0.1,60.0,8.0,1.0,0.6,1,1,1,NO,0,NO,NO,1,NO,NO,"Instantaneous, without keep-ho",9.0,91.1,0.95,1.0,1.0,NO,NO,NO,inapplicable,0.0,0.0,0.0,0.00,None,0.0,Combi boiler,600.0,0.0,NO,NO,9.0,NO,NO,NO,0.0,0.0,0.0,0.0,1.0,67.050,2.0,54.270,0.280,0.0,0.0,0.0,0.0,Dublin 12,Semi-detached house,Mains Gas,Mains Gas
18,19,2005,Existing,Sale,2021-01-04,NO,0,30,8,132.19,B3,88.000,2,40.33,44.95,44.95,12.78,1.85,17.67,0.00,Pitch Roof-Insul.on Ceiling,43.050,0.37,0.26,0.26,2.00,3.00,0.0,0.15,Medium,Masonry,No,90.0,4.0,NO,0.00,1.0,0.0,1.0,0.0,NO,Natural vent.,0.00,0.0,90.1,0.95,0.1,30.0,8.0,1.0,0.0,2,1,1,YES,0,NO,NO,1,NO,NO,None,9.0,90.1,0.95,28.0,1.0,YES,NO,YES,inapplicable,110.0,0.0,0.6,1.00,Factory Insulated,30.0,Boiler with uninsulated primar,0.0,0.0,NO,NO,9.0,NO,NO,NO,0.0,0.0,0.0,0.0,2.0,88.000,1.0,40.330,0.370,0.0,0.0,0.0,0.0,Dublin 11,Mid-terrace house,Mains Gas,Mains Gas
23,24,1994,Existing,Sale,2021-01-04,NO,0,47,7,257.62,D1,30.620,1,34.00,0.00,0.00,4.97,1.68,51.08,0.00,NaN,NaN,0.55,0.00,0.00,2.80,1.36,0.0,0.15,Medium,Masonry,No,100.0,4.0,NO,0.00,1.0,0.0,2.0,0.0,YES,Natural vent.,0.00,0.0,100.0,1.00,0.0,0.0,1.0,1.0,0.0,3,1,0,NO,0,NO,NO,0,NO,NO,None,6.0,100.0,1.00,1.0,1.0,YES,NO,NO,inapplicable,135.0,0.0,0.6,1.00,Factory Insulated,25.0,Electric immersion heater,0.0,0.0,NO,NO,6.0,NO,NO,NO,0.0,0.0,0.0,0.0,1.0,30.620,2.0,34.000,0.550,0.0,0.0,0.0,0.0,Dublin 7,Mid-floor apartment,Electricity,Electricity
24,25,1900,Existing,Grant Support,2021-01-04,NO,0,42,4,279.50,D2,55.432,2,52.74,27.35,27.72,5.99,1.99,34.26,0.00,Pitch Roof-Insul.on Ceiling,18.618,1.33,1.65,0.51,2.03,2.40,0.0,0.15,Medium-low,Masonry,Yes (Unsealed),88.0,4.0,NO,0.00,1.0,0.0,4.0,0.0,NO,Natural vent.,0.00,0.0,91.0,1.00,0.1,30.0,8.0,1.0,0.0,3,1,1,YES,0,NO,NO,1,NO,NO,"Instantaneous, without keep-ho",9.0,91.0,1.00,1.0,1.0,NO,NO,NO,inapplicable,0.0,0.0,0.0,0.00,None,0.0,Combi boiler,600.0,0.0,NO,NO,9.0,NO,NO,NO,0.0,0.0,0.0,0.0,2.0,55.432,2.0,52.741,1.235,0.0,0.0,0.0,0.0,Dublin 3,End of terra

In [79]:
# Finding columns that have only ONE value for every home.
# A column that never changes can't help the model tell homes apart, so we remove it.
# (nunique counts the different values, ignoring missing ones)

constant_columns = []

for column in df_dublin_ber.columns:
    if df_dublin_ber[column].nunique() <= 1:
        constant_columns.append(column)
        print(column, '->', df_dublin_ber[column].unique()[:3])

print('Constant columns:', len(constant_columns))
print('Column names:', constant_columns)

KeepHotFacility -> <StringArray>
['inapplicable                  ']
Length: 1, dtype: str
num_wind_turbines -> [0.]
Constant columns: 2
Column names: ['KeepHotFacility', 'num_wind_turbines']


In [80]:
# Removing them

print('Columns before:', df_dublin_ber.shape[1])
df_dublin_ber = df_dublin_ber.drop(columns=constant_columns)
print('Columns after: ', df_dublin_ber.shape[1])

Columns before: 103
Columns after:  101


In [81]:
# Removing the id columns.
# They are just code numbers (e.g. county_id 6, fuel_id 9). We already have the
# names in CountyName, DwellingTypeDescr, sh_main_fuel and wh_main_fuel.
# We keep ber_id: it is not a feature, but we need it to know which home is which.

id_columns = []

for column in df_dublin_ber.columns:
    if column.endswith('_id') and column != 'ber_id':
        id_columns.append(column)

print('Id columns:', id_columns)

print('Columns before:', df_dublin_ber.shape[1])
df_dublin_ber = df_dublin_ber.drop(columns=id_columns)
print('Columns after: ', df_dublin_ber.shape[1])

Id columns: ['county_id', 'dwelling_type_id', 'sh_main_fuel_id', 'wh_main_fuel_id']
Columns before: 101
Columns after:  97


In [84]:
# Renaming CountyName to dublin_district.
# Homes recorded as 'Co. Dublin' don't say which district they are in,
# so they become 'Dublin (no district)' (the same as in our earlier Dublin files).

df_dublin_ber = df_dublin_ber.rename(columns={'CountyName': 'dublin_district'})

df_dublin_ber['dublin_district'] = df_dublin_ber['dublin_district'].replace('Co. Dublin', 'Dublin (no district)')

df_dublin_ber['dublin_district'].value_counts()

dublin_district
Dublin (no district)    48327
Dublin 15               13465
Dublin 18               12075
Dublin 24               11841
Dublin 12                7457
Dublin 8                 7260
Dublin 11                7119
Dublin 22                6906
Dublin 9                 6903
Dublin 7                 6265
Dublin 13                5829
Dublin 4                 5789
Dublin 16                5670
Dublin 14                5362
Dublin 3                 5120
Dublin 5                 5034
Dublin 1                 4556
Dublin 6                 4106
Dublin 6W                2808
Dublin 10                2631
Dublin 17                2363
Dublin 2                 2246
Dublin 20                2063
Name: count, dtype: int64

In [85]:
# Putting the columns in a clear order:
# id -> date -> county -> important categories -> other categories -> numbers -> targets

id_column = ['ber_id']
date_column = ['DateOfAssessment']
county_column = ['dublin_district']
important_categorical = ['DwellingTypeDescr', 'sh_main_fuel', 'wh_main_fuel']
target_columns = ['BerRating', 'EnergyRating']

already_placed = id_column + date_column + county_column + important_categorical + target_columns

# Every other column goes into "numbers" or "categories" depending on its type
other_categorical = []
numeric_columns = []

for column in df_dublin_ber.columns:
    if column in already_placed:
        continue
    if pd.api.types.is_numeric_dtype(df_dublin_ber[column]):
        numeric_columns.append(column)
    else:
        other_categorical.append(column)

new_order = (id_column + date_column + county_column
             + important_categorical + other_categorical
             + numeric_columns + target_columns)

print('Columns before:', df_dublin_ber.shape[1], ' after:', len(new_order))   # must be the same
df_dublin_ber = df_dublin_ber[new_order]

df_dublin_ber.head()

Columns before: 97  after: 97


,ber_id,DateOfAssessment,dublin_district,DwellingTypeDescr,sh_main_fuel,wh_main_fuel,TypeofRating,PurposeOfRating,MultiDwellingMPRN,PredominantRoofType,ThermalMassCategory,StructureType,SuspendedWoodenFloor,PermeabilityTest,DraftLobby,VentilationMethod,CHBoilerThermostatControlled,OBBoilerThermostatControlled,OBPumpInsideDwelling,WarmAirHeatingSystem,UndergroundHeating,CombiBoiler,StorageLosses,ManuLossFactorAvail,ElecImmersionInSummer,InsulationType,PrimaryCircuitLoss,CylinderStat,CombinedCylinder,SolarHotWaterHeating,SWHPumpSolarPowered,ChargingBasisHeatConsumed,Year_of_Construction,TGDLEdition,GroundFloorArea(sq m),NoStoreys,WallArea,RoofArea,FloorArea,WindowArea,DoorArea,LivingAreaPercent,RoomInRoofArea,PredominantRoofTypeArea,UValueWall,UValueRoof,UValueFloor,UValueWindow,UvalueDoor,GroundFloorUValue,ThermalBridgingFactor,PercentageDraughtStripped,NoOfSidesSheltered,PermeabilityTestResult,NoOfChimneys,NoOfOpenFlues,NoOfFansAndVents,NoOfFluelessGasFires,FanPowerManuDeclaredValue,HeatExchangerEff,HSMainSystemEfficiency,HSEffAdjFactor,HSSupplHeatFraction,HSSupplSystemEff,SupplSHFuel,SHRenewableResources,TempAdjustment,HeatSystemControlCat,HeatSystemResponseCat,NoCentralHeatingPumps,NoOilBoilerHeatingPumps,NoGasBoilerHeatingPumps,WHMainSystemEff,WHEffAdjFactor,SupplWHFuel,WHRenewableResources,WaterStorageVolume,DeclaredLossFactor,TempFactorUnadj,TempFactorMultiplier,InsulationThickness,CombiBoilerAddLoss,ElecConsumpKeepHot,IsSolarPVBatteryPresent,SolarBatteryCapacity,SolarPVEnergyOutput,WindTurbineEnergyOutput,num_floors,total_floor_area,num_walls,total_wall_area,mean_wall_uvalue,num_other_energy,num_group_boilers,num_pv_arrays,BerRating,EnergyRating
13,14,2021-01-03,Dublin 12,Semi-detached house,Mains Gas,Mains Gas,Existing,Grant Support,NO,Pitch Roof-Insul.on Ceiling,High,Masonry,No,NO,NO,Natural vent.,NO,NO,NO,NO,NO,"Instantaneous, without keep-ho",NO,NO,NO,None,Combi boiler,NO,NO,NO,NO,NO,1930,0,67.050,1,54.27,67.05,67.05,9.21,2.18,35.44,0.0,67.050,0.29,0.19,0.73,1.10,1.71,0.0,0.15,88.0,2.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,91.1,0.95,0.1,60.0,8.0,1.0,0.6,1,1,1,0,1,91.1,0.95,1.0,1.0,0.0,0.0,0.0,0.0,0.0,600.0,0.0,0.0,0.0,0.0,0.0,1.0,67.050,2.0,54.270,0.280,0.0,0.0,0.0,216.42,C3
18,19,2021-01-04,Dublin 11,Mid-terrace house,Mains Gas,Mains Gas,Existing,Sale,NO,Pitch Roof-Insul.on Ceiling,Medium,Masonry,No,NO,NO,Natural vent.,YES,NO,NO,NO,NO,None,YES,NO,YES,Factory Insulated,Boiler with uninsulated primar,NO,NO,NO,NO,NO,2005,0,88.000,2,40.33,44.95,44.95,12.78,1.85,17.67,0.0,43.050,0.37,0.26,0.26,2.00,3.00,0.0,0.15,90.0,4.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,90.1,0.95,0.1,30.0,8.0,1.0,0.0,2,1,1,0,1,90.1,0.95,28.0,1.0,110.0,0.0,0.6,1.0,30.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,88.000,1.0,40.330,0.370,0.0,0.0,0.0,132.19,B3
23,24,2021-01-04,Dublin 7,Mid-floor apartment,Electricity,Electricity,Existing,Sale,NO,NaN,Medium,Masonry,No,NO,YES,Natural vent.,NO,NO,NO,NO,NO,None,YES,NO,NO,Factory Insulated,Electric immersion heater,NO,NO,NO,NO,NO,1994,0,30.620,1,34.00,0.00,0.00,4.97,1.68,51.08,0.0,NaN,0.55,0.00,0.00,2.80,1.36,0.0,0.15,100.0,4.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,100.0,1.00,0.0,0.0,1.0,1.0,0.0,3,1,0,0,0,100.0,1.00,1.0,1.0,135.0,0.0,0.6,1.0,25.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,30.620,2.0,34.000,0.550,0.0,0.0,0.0,257.62,D1
24,25,2021-01-04,Dublin 3,End of terrace house,Mains Gas,Mains Gas,Existing,Grant Support,NO,Pitch Roof-Insul.on Ceiling,Medium-low,Masonry,Yes (Unsealed),NO,NO,Natural vent.,YES,NO,NO,NO,NO,"Instantaneous, without keep-ho",NO,NO,NO,None,Combi boiler,NO,NO,NO,NO,NO,1900,0,55.432,2,52.74,27.35,27.72,5.99,1.99,34.26,0.0,18.618,1.33,1.65,0.51,2.03,2.40,0.0,0.15,88.0,4.0,0.0,1.0,0.0,4.0,0.0,0.0,0.0,91.0,1.00,0.1,30.0,8.0,1.0,0.0,3,1,1,0,1,91.0,1.00,1.0,1.0,0.0,0.0,0.0,0.0,0.0,600.0,0.0,0.0,0.0,0.0,0.0,2.0,55.432,2.0,52.741,1.235,0.0,0.0,0.0,279.50,D2
26,27,2021-01-04,Dublin 9,Mid-terrace house,Mains Gas,Mains Gas,Existing,Sale,NO,Pitch Roof-Insul.on Rafter,Medium,Masonry,No,NO,NO,Natural vent.,YES,NO,NO,NO,NO,None,YES,NO,NO,Factory Insulated,